# 5-fold cross-validation on Green (FNC v2) — Kaggle

One run = one recipe on the 5 folds (`splits/green_cv{k}.json`). For each fold: training (100 epochs, seed 42, no
early stopping), then predictions on the held-out images with `last.pt` (epoch 100) → `pred_val.npz`.

This run: `RECIPE = "light"`, c-ResUNet-light → `runs/light_cv{k}/` (`runs/cresunet_light_cv{k}/` in this repository).

Accelerator: **GPU T4 x2**. Inputs: the folders `cellseg-code` (code) and `fnc-green-data` (images and cleaned masks), found anywhere
under `/kaggle/input`. Short check with `PROBE = True`, then the full run with `PROBE = False` and
**Save Version → Save & Run All (Commit)**.

In [1]:
# 1) GPU available?
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Settings → Accelerator → 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128
GPU: Tesla T4


In [2]:
# 2) Find code and data under /kaggle/input, copy the code to /kaggle/working
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def find_one(pattern):
    '''First path under INPUT matching the pattern (Kaggle can nest dataset folders in different ways).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} — did you add both datasets with 'Add Input'?"
    return hits[0]

CODE_SRC = find_one("models/c_res_unet_logits.py").parent.parent
IMAGES = find_one("green/trainval/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
print("code:  ", CODE_SRC)
print("images:", IMAGES, "→", len(list(IMAGES.glob("*.png"))), "png (expected 212)")
print("masks: ", MASKS, "→", len(list(MASKS.glob("*.png"))), "png (expected 213: 128.png has no image)")

CODE = WORK / "code"   # /kaggle/input is read-only
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)
os.chdir(CODE)
assert all((CODE / f"splits/green_cv{k}.json").exists() for k in range(5)), "old cellseg-code: upload the new version (with splits/green_cv*.json)"
print("working dir:", os.getcwd())

code:   /kaggle/input/datasets/alixram/cell-segmentation/cellseg-code
images: /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/green/trainval/images → 212 png (expected 212)
masks:  /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/cleaned_masks/trainval/Green/masks → 213 png (expected 213: 128.png has no image)
working dir: /kaggle/working/code


## 3) Parameters

- `RECIPE`: `"v2"` / `"v7"`: c-ResUNet of Morelli et al. (2021) without / with weight maps; `"light"` /
  `"light_wm"`: c-ResUNet-light without / with weight maps.
- `PROBE = True`: short check (fold 0, 1 epoch, 8 training and 4 validation images); `PROBE = False`: full run.
- `FOLDS`: all 5; fewer only to resume an interrupted run.

In [3]:
RECIPE = "light"   # "v2", "v7", "light" or "light_wm"
PROBE = False
FOLDS = [0, 1, 2, 3, 4]

RECIPES = {"v2": ("green_cv", []), "v7": ("green_wm_cv", ["--weight_maps"]),          # reference network (done)
           "light": ("light_cv", ["--small_rf"]), "light_wm": ("light_wm_cv", ["--small_rf", "--weight_maps"])}
assert RECIPE in RECIPES, RECIPE
PREFIX, EXTRA = RECIPES[RECIPE]
EPOCHS = 1 if PROBE else 100
LIMITS = ["--train_limit", "8", "--val_limit", "4"] if PROBE else []
if PROBE:
    FOLDS = [0]
    PREFIX = "probe_" + PREFIX
print(RECIPE, "| folds", FOLDS, "| epochs", EPOCHS, "| out runs/" + PREFIX + "{k}")

light | folds [0, 1, 2, 3, 4] | epochs 100 | out runs/light_cv{k}


## 4) Training and predictions, fold by fold

Stops at the first error. The predictions use `last.pt` (epoch 100), not `best.pt`, which is chosen on the held-out
fold itself.

In [4]:
import subprocess
import sys
import time

def run(cmd):
    '''Runs a command, streams its output, stops on error.'''
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, f"failed (exit code {p.returncode})"

t_start = time.time()
for k in FOLDS:
    t0 = time.time()
    out = WORK / "runs" / f"{PREFIX}{k}"
    split = f"splits/green_cv{k}.json"
    run([sys.executable, "-m", "train.training", "--images_dir", IMAGES, "--masks_dir", MASKS, "--split_json", split,
         "--out_dir", out, "--epochs", EPOCHS, "--patience", EPOCHS, "--num_workers", 4, *EXTRA, *LIMITS])
    run([sys.executable, "-m", "evaluation.predict", "--ckpt", out / "last.pt", "--images_dir", IMAGES, "--masks_dir", MASKS,
         "--split_json", split, "--subset", "val", "--out", out / "pred_val.npz"])
    print(f"=== fold {k} done in {(time.time() - t0) / 60:.1f} min (total {(time.time() - t_start) / 60:.1f} min)", flush=True)

$ /usr/bin/python3 -m train.training --images_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/green/trainval/images --masks_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/cleaned_masks/trainval/Green/masks --split_json splits/green_cv0.json --out_dir /kaggle/working/runs/light_cv0 --epochs 100 --patience 100 --num_workers 4 --small_rf
dati: 169 train, 43 val, caricati in 38 s | mean 0.0492 std 0.0700 | device cuda, amp True
ep   1 | lr 1.00e-03 | train loss 0.5831 | val loss 0.5403 dice 0.5468 | 70 s *
ep   2 | lr 1.00e-03 | train loss 0.3865 | val loss 0.5822 dice 0.5072 | 31 s 
ep   3 | lr 9.99e-04 | train loss 0.3671 | val loss 0.4281 dice 0.6408 | 34 s *
ep   4 | lr 9.98e-04 | train loss 0.3443 | val loss 0.4554 dice 0.6159 | 33 s 
ep   5 | lr 9.96e-04 | train loss 0.3476 | val loss 0.4749 dice 0.6070 | 33 s 
ep   6 | lr 9.94e-04 | train loss 0.3296 | val loss 0.8273 dice 0.2241 | 34 s 
ep   7 | lr 9.91e-04 | train loss 0.3289 | val loss 0.3

In [5]:
# 5) Summary: one row per fold
import json
import numpy as np
import pandas as pd

rows = []
for k in FOLDS:
    out = WORK / "runs" / f"{PREFIX}{k}"
    log = pd.read_csv(out / "log.csv")
    cfg = json.load(open(out / "config.json"))
    pred = np.load(out / "pred_val.npz")
    held = json.load(open(f"splits/green_cv{k}.json"))["val"]
    assert list(pred["names"]) == held, f"fold {k}: predicted images differ from the held-out fold"
    rows.append({"fold": k, "epochs": len(log), "weight_maps": cfg.get("weight_maps", False), "n_train": cfg["n_train"],
                 "held_out": len(held), "best_epoch": int(log.epoch[log.val_dice_global.idxmax()]),
                 "best_dice": log.val_dice_global.max(), "last_dice": log.val_dice_global.iloc[-1],
                 "min_per_epoch": log.time_s.mean() / 60})
print(pd.DataFrame(rows).round(4).to_string(index=False))

 fold  epochs  weight_maps  n_train  held_out  best_epoch  best_dice  last_dice  min_per_epoch
    0     100        False      169        43          52     0.7187     0.7132         0.5598
    1     100        False      169        43          83     0.7716     0.7677         0.5499
    2     100        False      170        42          59     0.7685     0.7631         0.5552
    3     100        False      170        42          59     0.7678     0.7655         0.5565
    4     100        False      170        42          62     0.7403     0.7255         0.5466
